In [7]:
import dolfinx as dfx
import multiphenicsx.fem
from mpi4py import MPI
from petsc4py.PETSc import ScalarType
import numpy as np
import ufl
from solver import NonlinearBlockProblem, NewtonBlockSolver
import matplotlib.pyplot as plt

n= 20
mesh = dfx.mesh.create_rectangle(MPI.COMM_WORLD, [(0,0),(1,300)])
mesh = dfx.mesh.create_unit_square(MPI.COMM_WORLD, n, 10*n)
cdim = mesh.topology.dim
fdim = mesh.topology.dim - 1

# Define boundaries
boundaries = [(1, lambda x: np.isclose(x[0], 0)),
                (2, lambda x: np.isclose(x[0], 1)),]
left_facets = dfx.mesh.locate_entities(mesh, fdim, boundaries[0][1]) 
right_facets = dfx.mesh.locate_entities(mesh, fdim, boundaries[1][1])
facet_indices, facet_markers = [], []
for (marker, locator) in boundaries:
    facets = dfx.mesh.locate_entities(mesh, fdim, locator)
    facet_indices.append(facets)
    facet_markers.append(np.full(len(facets), marker))
facet_indices = np.array(np.hstack(facet_indices), dtype=np.int32)
facet_markers = np.array(np.hstack(facet_markers), dtype=np.int32)
sorted_facets = np.argsort(facet_indices)
facet_tag = dfx.mesh.MeshTags(mesh, fdim, facet_indices[sorted_facets], facet_markers[sorted_facets])

# Define domains
domains = [
    (1, lambda x: x[0]<=0.2),
    (2, lambda x: x[0]>=0.2),
]
left_cells = dfx.mesh.locate_entities(mesh, cdim, domains[0][1]) 
right_cells = dfx.mesh.locate_entities(mesh, cdim, domains[1][1])
cell_indices, cell_markers = [], []
for (marker, locator) in domains:
    cells = dfx.mesh.locate_entities(mesh, cdim, locator)
    cell_indices.append(cells)
    cell_markers.append(np.full(len(cells), marker))
cell_indices = np.array(np.hstack(cell_indices), dtype=np.int32)
cell_markers = np.array(np.hstack(cell_markers), dtype=np.int32)
sorted_cells = np.argsort(cell_indices)
cell_tag = dfx.mesh.MeshTags(mesh, cdim, cell_indices[sorted_cells], cell_markers[sorted_cells])

# Define function spaces
PHI_E = dfx.fem.FunctionSpace(mesh, ('CG',1))
PHI_S = PHI_E.clone()
LM_IAPP = PHI_E.clone()
phi_e, phi_s, lm_iapp = dfx.fem.Function(PHI_E, name='phi_e'), dfx.fem.Function(PHI_S, name='phi_s'), dfx.fem.Function(LM_IAPP,name='lm_iapp')
dphi_e, dphi_s, dlm_iapp = ufl.TrialFunction(PHI_E), ufl.TrialFunction(PHI_S), ufl.TrialFunction(LM_IAPP)
v_phi_e, v_phi_s, v_lm_iapp = ufl.TestFunction(PHI_E), ufl.TestFunction(PHI_S), ufl.TestFunction(LM_IAPP)
# Define measures
dx = ufl.Measure('dx', mesh, subdomain_data=cell_tag)
ds = ufl.Measure('ds', mesh, subdomain_data=facet_tag)
# Define constants 
I_app = dfx.fem.Constant(mesh, ScalarType(1))
V_app = dfx.fem.Constant(mesh, ScalarType(1))
switch = dfx.fem.Constant(mesh, ScalarType(0))


# Define WF
reaction = 1e-5*2*ufl.sinh(0.5*94685/(8.3*298)*(phi_s-phi_e))
residuals = [
    ufl.inner(ufl.grad(phi_e),ufl.grad(v_phi_e)) * dx - lm_iapp*v_phi_e * ds(2) - reaction*v_phi_e*dx(1),
    1e2*ufl.inner(ufl.grad(phi_s),ufl.grad(v_phi_s)) * dx(1) + reaction*v_phi_s*dx(1),
    (1-switch)*(I_app-lm_iapp)*v_lm_iapp*ds(2) + switch*(V_app-phi_e)*v_lm_iapp*ds(2)
]
jacobian = [[ufl.derivative(r_i, u_i, du_i) for u_i, du_i in zip([phi_e, phi_s, lm_iapp],[dphi_e, dphi_s, dlm_iapp])] for r_i in residuals]
bc = dfx.fem.dirichletbc(dfx.fem.Constant(mesh, ScalarType(0)), dfx.fem.locate_dofs_topological(PHI_S, fdim, left_facets), PHI_S)
restrictions = [
    multiphenicsx.fem.DofMapRestriction(PHI_E.dofmap, np.arange(0, PHI_E.dofmap.index_map.size_local+PHI_E.dofmap.index_map.num_ghosts)),
    multiphenicsx.fem.DofMapRestriction(PHI_S.dofmap, dfx.fem.locate_dofs_topological(PHI_S, cdim, left_cells)), 
    multiphenicsx.fem.DofMapRestriction(LM_IAPP.dofmap, dfx.fem.locate_dofs_topological(LM_IAPP, fdim, right_facets))
]

# Solve
problem = NonlinearBlockProblem(residuals, [phi_e, phi_s, lm_iapp], [bc], jacobian, restrictions)
solver = NewtonBlockSolver(mesh.comm, problem)
solver.solve()






 Num MPI tasks = 1

 Num OpenMP threads = 1


BoomerAMG SETUP PARAMETERS:

 Max levels = 25
 Num levels = 9

 Strength Threshold = 0.250000
 Interpolation Truncation Factor = 0.000000
 Maximum Row Sum Threshold for Dependency Weakening = 0.900000

 Coarsening Type = Falgout-CLJP 
 measures are determined locally


 No global partition option chosen.

 Interpolation = modified classical interpolation

Operator Matrix Information:

             nonzero            entries/row          row sums
lev    rows  entries sparse   min  max     avg      min         max
  0    5427    49903  0.002     5   14     9.2  -5.000e-03   1.000e+01
  1    2500    27714  0.004     4   18    11.1  -7.838e-13   1.990e+01
  2    1250    13764  0.009     4   18    11.0  -3.482e-13   4.786e+01
  3     625     7295  0.019     4   23    11.7  -7.244e-15   8.890e+01
  4     330     6204  0.057     8   32    18.8  -9.118e-15   1.155e+02
  5     134     2932  0.163     6   36    21.9   3.080e-15   1.418e+02
  6     

(9, True)

In [8]:
import pyvista
pyvista.start_xvfb()
pyvista.set_jupyter_backend('panel')
cells, types, x = dfx.plot.create_vtk_mesh(PHI_E)
grid = pyvista.UnstructuredGrid(cells, types, x)
grid.point_data["u"] = phi_e.x.array.real
grid.set_active_scalars("u")
plotter = pyvista.Plotter()
# plotter.add_mesh(grid, show_edges=True)
warped = grid.warp_by_scalar()
plotter.add_mesh(warped, show_edges=True)
plotter.show()